### Análise Exploratória de Dados: `food_avaliacoes_produto`
**Visão Geral:** Total de Registros, Colunas, Estrutura`(Schema)`.<br>
**Nulos:** Total de registros nulos por colunas.<br>
**Primaky Key:** Total de PK nulos, PK duplicadas e Registros Duplicados.<br>
**Data Máxima e Mínima:** Range de datas da tabela.<br>
**Valores Distintos:** Valores Distintos por colunas.<br>
**Valores de Avaliações da Coluna notas:** Range de valores de avaliações atribuídos para a coluna notas.<br>

In [0]:
import os
from dotenv import load_dotenv

load_dotenv()

client_id = os.getenv("CLIENT_ID")
tenant_id = os.getenv("TENANT_ID")
client_secret = os.getenv("CLIENT_SECRET")
storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")
container_name = os.getenv("CONTAINER_NAME")

adls_options = {
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

In [0]:
food_avaliacoes_produto_path = (
    f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/"
    "batch-data/food_avaliacoes_produto.csv"
)

df_food_avaliacoes_produto_raw = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(food_avaliacoes_produto_path)
)


# VISÃO GERAL
from pyspark.sql import functions as F

def overview(df, dataset_name):
    print(f"\n=== VISÃO GERAL: {dataset_name} ===")

    total_records = df.count()
    total_columns = len(df.columns)

    print(f"Total de registros: {total_records}")
    print(f"Total de colunas: {total_columns}")

    print("\nSchema:")
    df.printSchema()

    print("\nPrimeiros registros:")
    display(df.limit(5))


# NULOS
def null_analysis(df, dataset_name):
    print(f"\n=== VALORES NULOS: {dataset_name} ===")

    total_records = df.count()

    null_counts = df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ])

    display(null_counts)

# PRIMARY KEY
def primary_key_analysis(df, primary_key, dataset_name):
    print(f"\n=== CHAVE PRIMÁRIA: {dataset_name} ===")
    print(f"Chave: {primary_key}")

    null_keys = df.filter(
        F.col(primary_key).isNull()
    ).count()

    duplicated_keys = (
        df.groupBy(primary_key)
          .count()
          .filter(F.col("count") > 1)
    )

    total_duplicated_keys = duplicated_keys.count()

    total_records = df.count()
    total_unique = df.dropDuplicates([primary_key]).count()
    total_duplicates = total_records - total_unique

    print(f"Chaves nulas: {null_keys}")
    print(f"Chaves duplicadas: {total_duplicated_keys}")
    print(f"Registros duplicados: {total_duplicates}")

    if total_duplicated_keys > 0:
        print("\nChaves duplicadas:")
        display(duplicated_keys)


# DATAS MÁXIMAS E MÍNIMAS
def date_range_analysis(df, date_column_name, dataset_name):
    print(f"\n=== PERÍODO DOS DADOS: {dataset_name} => COLUNA: {date_column_name} ===")
    
    result = df.select(
        F.min(F.col(date_column_name)).alias("data_minima"),
        F.max(F.col(date_column_name)).alias("data_maxima")
    )
    
    display(result)


# VALORES DISTINTOS
def distinct_analysis(df, dataset_name):
    print(f"\n=== VALORES DISTINTOS: {dataset_name} ===")

    distinct_counts = df.select([
        F.countDistinct(F.col(c)).alias(c)
        for c in df.columns
    ])

    display(distinct_counts)


# VALORES AVALIAÇÕES COLUNA NOTA
def values_analysis(df, column_name, dataset_name):
    print(f"\n=== VALORES DISTINTOS: {dataset_name} ===")
    print(f"Coluna: {column_name}")

    result = (
        df.groupBy(column_name)
          .count()
          .orderBy(column_name)
    )

    display(result)

In [0]:
overview(
    df_food_avaliacoes_produto_raw,
    "food_avaliacoes_produto"
)

null_analysis(
    df_food_avaliacoes_produto_raw,
    "food_avaliacoes_produto"
)

primary_key_analysis(
    df_food_avaliacoes_produto_raw,
    "id_avaliacao",
    "food_avaliacoes_produto"
)

date_range_analysis(
    df_food_avaliacoes_produto_raw,
    "dt_avaliacao",
    "food_avaliacoes_produto"
)

values_analysis(
    df_food_avaliacoes_produto_raw,
    "nota",
    "food_avaliacoes_produto"
)

distinct_analysis(
    df_food_avaliacoes_produto_raw,
    "food_avaliacoes_produto"
)
